# SpectraShift Week 3: VICReg pilots
Run the two predeclared M3 learning-rate pilots against the frozen Week 2 dataset. Attach the updated SpectraShift source and `spectrashift-week2-frozen`. Select GPU T4 x2; this code intentionally uses one T4. Internet is not required.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/spectrashift-week3-pilots')
WORK.mkdir(parents=True, exist_ok=True)
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift').is_dir()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one SpectraShift source bundle, found {bundles}'
    project = WORK / 'source'
    shutil.unpack_archive(str(bundles[0]), str(project))
    projects = [project]
staging_summaries = sorted(INPUT.rglob('staging_summary.json'))
partition_manifests = sorted(INPUT.rglob('partitions.parquet'))
normalizations = sorted(INPUT.rglob('normalization.json'))
freeze_summaries = sorted(INPUT.rglob('freeze_summary.json'))
assert len(projects) == 1, f'Expected one source tree, found {projects}'
assert len(staging_summaries) == 1, f'Expected one frozen staging summary, found {staging_summaries}'
assert len(partition_manifests) == 1, f'Expected one frozen partition manifest, found {partition_manifests}'
assert len(normalizations) == 1, f'Expected one normalization artifact, found {normalizations}'
assert len(freeze_summaries) == 1, f'Expected one freeze summary, found {freeze_summaries}'
PROJECT = projects[0]
STAGED = staging_summaries[0].parent
MANIFEST = partition_manifests[0]
NORMALIZATION = normalizations[0]
FREEZE_SUMMARY = freeze_summaries[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
print({'project': str(PROJECT), 'staged': str(STAGED), 'manifest': str(MANIFEST), 'work': str(WORK)})


In [ ]:
staging = json.loads(staging_summaries[0].read_text())
normalization = json.loads(NORMALIZATION.read_text())
freeze = json.loads(FREEZE_SUMMARY.read_text())
assert staging['complete_patches'] == 50200 and staging['incomplete_patches'] == 0, staging
assert staging['archive_part_sha256']['BigEarthNet-S2.tar.gzaa'] == 'a44e4c9d8dde1affd7107640dd0070b811ca848205363f8dd8f75bebccea0ed3', staging
assert staging['archive_part_sha256']['BigEarthNet-S2.tar.gzab'] == 'e0bfe57038e07ff09add42e5bff108d221da10a6e45533fd2eb4ac05e6d90dc2', staging
assert normalization['sha256'] == '3b1d191beccde5b85fdced38c81984377e6fb3171683a6df6a9254c38f2a9a05', normalization
assert freeze['training_approved'] and freeze['manifest_sha256'] == '0b36a0c6c55f34a8963719af725096dde5ab1dbab72968c13639e31d1099000a', freeze
runtime_configs = []
for filename in ('week3_m3_lr1e4.yaml', 'week3_m3_lr3e4.yaml'):
    config = yaml.safe_load((PROJECT / 'configs/ssl' / filename).read_text())
    config['data']['manifest_path'] = str(MANIFEST)
    config['data']['staged_root'] = str(STAGED)
    config['data']['normalization_path'] = str(NORMALIZATION)
    config['data']['freeze_summary_path'] = str(FREEZE_SUMMARY)
    config['run']['output_dir'] = str(WORK / config['run']['id'])
    config['run']['ledger_path'] = str(WORK / 'runs.jsonl')
    runtime = WORK / filename
    runtime.write_text(yaml.safe_dump(config, sort_keys=False))
    runtime_configs.append(runtime)
print([str(path) for path in runtime_configs])


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select GPU T4 x2 before running'
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
device_arch = f'sm_{major}{minor}'
supported_arches = torch.cuda.get_arch_list()
print({'gpu': gpu_name, 'device_arch': device_arch, 'pytorch_arches': supported_arches})
assert 'T4' in gpu_name, f'Select GPU T4 x2, found {gpu_name}'
assert device_arch in supported_arches, f'{device_arch} is unsupported by this PyTorch build'


In [ ]:
from spectrashift.train.ssl import train_ssl
ssl_summaries = []
for runtime in runtime_configs:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    result = train_ssl(runtime)
    ssl_summaries.append(result)
    print(json.dumps(result, indent=2))


In [ ]:
from spectrashift.train.probe import probe_ssl
probe_summaries = []
for runtime, ssl_summary in zip(runtime_configs, ssl_summaries, strict=True):
    torch.cuda.empty_cache()
    probe = probe_ssl(runtime, ssl_summary['checkpoint'])
    probe_summaries.append(probe)
    print(json.dumps(probe, indent=2))


In [ ]:
from spectrashift.train.pilots import select_week4_pilot
selection = select_week4_pilot(ssl_summaries, probe_summaries)
summary = {'week3_complete': bool(selection['week4_approved']), 'week4_selection': selection, 'ssl_pilots': ssl_summaries, 'linear_probes': probe_summaries, 'evaluation_labels_loaded': False}
summary_path = WORK / 'week3_run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
assert selection['week4_approved'], 'Both pilots failed; run the declared 3e-5 diagnostic before Week 4'
